In [ ]:
import numpy as np
import pandas as pd
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import distinctipy
import pickle
from collections import Counter
from tqdm import tqdm 
import dask
import dask.array as da
import cv2
from pathlib import Path
from scipy.ndimage import binary_dilation
from joblib import Parallel, delayed
from tqdm_joblib import tqdm_joblib

In [ ]:
def create_color_dict(labels_unique):
    color_arr = distinctipy.get_colors(len(labels_unique))
    cell_type_color_dict = {}
    for i in range(len(labels_unique)):
        cell_type_color_dict[labels_unique[i]] = color_arr[i]
    return cell_type_color_dict

In [ ]:
radius = 100

In [ ]:
key = '/Multiplex 6/MP3-058-2'

In [ ]:
cell_clustering_base_path = 'Y:/coskun-lab/Aditi/01/Results/cell_segmentation_clustering' + key 

In [ ]:
pixel_clustering_base_path = 'Y:/coskun-lab/Aditi/01/Results/pixel_clustering/' + key + '/contour_removal_pixels_repeat'

In [ ]:
with open(cell_clustering_base_path + '/1x/segmentation_mask_merged_within_contour_for_pixel_clustering.pickle', 'rb') as f:
    cell_segmentation_mask = pickle.load(f)

In [ ]:
with open(pixel_clustering_base_path + '/wsi_labeled.pickle', 'rb') as f: 
    wsi_labeled = pickle.load(f)

In [ ]:
with open(pixel_clustering_base_path + '/cluster_color_dict.pickle', 'rb') as f: 
    cluster_color_dict = pickle.load(f)

In [ ]:
cell_cluster_location_df = pd.read_csv(cell_clustering_base_path + '/1x/cell_cluster_location_merged_unclassified_within_contour.csv')

In [ ]:
"""cell_cluster_location_df = cell_cluster_location_df.rename(columns={'Unnamed: 0': 'cell_index'})"""

In [ ]:
wsi_labeled_dask = da.from_array(wsi_labeled, chunks=(512, 512))

In [ ]:
cell_segmentation_mask_dask = da.from_array(cell_segmentation_mask, chunks=(512, 512))

In [ ]:
wsi_labeled_chunks = wsi_labeled_dask.blocks.ravel()
cell_segmentation_mask_chunks = cell_segmentation_mask_dask.blocks.ravel()

In [ ]:
tile_location_list = []
for i in range(0, cell_segmentation_mask.shape[0], 512):
    for j in range(0, cell_segmentation_mask.shape[1], 512):
        tile_location_list.append((i, j))

In [ ]:
cell_segmentation_mask_binary = np.zeros(cell_segmentation_mask.shape)
cell_segmentation_mask_binary[cell_segmentation_mask!=0] = 1
cell_segmentation_mask_binary = cv2.dilate(cv2.erode(cell_segmentation_mask_binary, np.ones((3, 3), np.uint8), iterations=1), np.ones((83, 83), np.uint8), iterations=1)

contours, _ = cv2.findContours(cell_segmentation_mask_binary.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contour = max(contours, key=cv2.contourArea)

contour_drawn = np.zeros((cell_segmentation_mask_binary.shape[0], cell_segmentation_mask_binary.shape[1]))
contour_drawn = cv2.drawContours(contour_drawn, [contour], -1, (255, 255, 255), -1)


In [ ]:
segmentation_mask_processed = cell_segmentation_mask.copy()
segmentation_mask_processed[contour_drawn == 0] = 0

In [ ]:
num_clusters = np.nanmax(wsi_labeled)

In [ ]:
cell_cluster_location_df = cell_cluster_location_df.drop(columns=['Unnamed: 0', 'index', 'index.1'])

In [ ]:
wsi_labeled_nan_removed = wsi_labeled.copy()
wsi_labeled_nan_removed[np.isnan(wsi_labeled_nan_removed)] = 0

In [ ]:
cell_id_list = list(cell_cluster_location_df['Cell ID'])

In [ ]:
Path.mkdir(Path('Y:/coskun-lab/Aditi/01/Results/cell_pixel_barcodes/' + key.split('/')[1]), exist_ok=True)
Path.mkdir(Path('Y:/coskun-lab/Aditi/01/Results/cell_pixel_barcodes' + key), exist_ok=True)
Path.mkdir(Path('Y:/coskun-lab/Aditi/01/Results/cell_pixel_barcodes' + key + '/radius' + str(radius)), exist_ok=True)

In [ ]:
pixel_barcode_base_path =  'Y:/coskun-lab/Aditi/01/Results/cell_pixel_barcodes' + key + '/radius' + str(radius)

#### Count pixels assigned to each cluster outside each cell

In [ ]:

from scipy.ndimage import distance_transform_edt
def compute_pixel_count_outside_cell(cell_idx):
    ys, xs = np.where(cell_segmentation_mask == cell_idx)
    y_min = max(ys.min() - radius, 0)
    y_max = min(ys.max() + radius + 1, cell_segmentation_mask.shape[0])
    x_min = max(xs.min() - radius, 0)
    x_max = min(xs.max() + radius + 1, cell_segmentation_mask.shape[1])
    
    cell_block = (cell_segmentation_mask[y_min:y_max, x_min:x_max] == cell_idx)
    label_block = wsi_labeled_nan_removed[y_min:y_max, x_min:x_max]
    dist = distance_transform_edt(~cell_block) #
    ring_mask = (dist <= radius) & (~cell_block)

    pixels_outside = label_block[ring_mask]

    pixels_outside_int = pixels_outside.astype(np.uint8)
    counts = np.bincount(pixels_outside_int, minlength=int(num_clusters)+1)[1:]
    
    return {i+1: v for i, v in enumerate(counts)}


In [ ]:
with tqdm_joblib(tqdm(desc="Processing cells", total=len(cell_id_list))) as progress_bar:
    pixel_counts_around_cells = Parallel(n_jobs=-1)(delayed(compute_pixel_count_outside_cell)(idx) for idx in cell_id_list)

In [ ]:
with open(pixel_barcode_base_path + '/pixel_counts_outside_cells.pickle', 'wb') as f: 
    pickle.dump(pixel_counts_around_cells, f)

In [ ]:
pixel_counts_around_cells_df = pd.DataFrame(pixel_counts_around_cells, columns=range(1, int(num_clusters)+1))
pixel_proportions_around_cells_df = pixel_counts_around_cells_df.copy()
pixel_proportions_around_cells_df = pixel_proportions_around_cells_df.div(pixel_counts_around_cells_df.sum(axis=1), axis=0)
pixel_proportions_around_cells_df['Cell ID'] = cell_cluster_location_df['Cell ID']
pixel_proportions_around_cells_df['X'] = cell_cluster_location_df['Y']
pixel_proportions_around_cells_df['Y'] = cell_cluster_location_df['X']
pixel_proportions_around_cells_df['Cell Type'] = cell_cluster_location_df['Cell Type']

In [ ]:
pixel_proportions_around_cells_na_removed_df = pixel_proportions_around_cells_df.dropna()

In [ ]:
pixel_proportions_around_cells_na_removed_df.to_csv(pixel_barcode_base_path + '/pixel_proportions_outside_cells.csv')